# Machine Learning: Gender Wage Gap Decomposition
**Author:** Antara Sudhir  
**Course:** AD688 — Applied Business Analytics  
**Goal:** Quantify how much of the gender wage gap is explained by occupation/industry segregation vs. unexplained factors, using regression decomposition and Random Forest feature importance.

**Data:** IPUMS USA 2024 ACS (`employed_only`, 1.49M workers with positive wages)  
**Models:** Linear Regression (baseline + full) and Random Forest Regressor  
**Outputs:** Precomputed CSVs saved to `data/processed/` for loading in `pages/ml_methods.qmd`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import sys
sys.path.insert(0, "..")
from analysis.utils import load_employment

df = load_employment()
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## Step 1: Baseline Regression (No Occupation/Industry)

Predicts `INCWAGE` using only `AGE`, `RACE_LABEL`, `STATE_NAME`, and `SEX_LABEL`.  
The gender coefficient here is the **raw wage gap** after controlling for basic demographics.

In [ ]:
baseline_df = df[['AGE', 'RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'INCWAGE']].copy()
baseline_encoded = pd.get_dummies(
    baseline_df, 
    columns=['RACE_LABEL', 'STATE_NAME', 'SEX_LABEL'], 
    drop_first=True
)

X_baseline = baseline_encoded.drop(columns=['INCWAGE'])
y_baseline = baseline_encoded['INCWAGE']

model_baseline = LinearRegression()
model_baseline.fit(X_baseline, y_baseline)

sex_col = [c for c in X_baseline.columns if 'SEX_LABEL' in c][0]
baseline_gender_coef = model_baseline.coef_[list(X_baseline.columns).index(sex_col)]

print(f"Baseline R²: {model_baseline.score(X_baseline, y_baseline):.4f}")
print(f"Gender coefficient ({sex_col}): ${baseline_gender_coef:,.2f}")
print(f"Interpretation: holding age, race, and state constant, men earn ${baseline_gender_coef:,.2f} more than women on average.")

## Step 2: Full Regression (Adding Occupation + Industry)

Adds broad `OCC_GROUP` (OCC // 100) and `IND_GROUP` (IND // 1000) to the model.  
The gender coefficient here is the **adjusted wage gap** after also controlling for job type.  
The difference between Step 1 and Step 2 coefficients = portion explained by occupational segregation.

In [ ]:
full_df = df[['AGE', 'RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC', 'IND', 'INCWAGE']].copy()
full_df['OCC_GROUP'] = (full_df['OCC'] // 100) * 100
full_df['IND_GROUP'] = (full_df['IND'] // 1000) * 1000

full_encoded = pd.get_dummies(
    full_df.drop(columns=['OCC', 'IND']),
    columns=['RACE_LABEL', 'STATE_NAME', 'SEX_LABEL', 'OCC_GROUP', 'IND_GROUP'],
    drop_first=True
)

X_full = full_encoded.drop(columns=['INCWAGE'])
y_full = full_encoded['INCWAGE']

model_full = LinearRegression()
model_full.fit(X_full, y_full)

sex_col_full = [c for c in X_full.columns if 'SEX_LABEL' in c][0]
full_gender_coef = model_full.coef_[list(X_full.columns).index(sex_col_full)]

print(f"Full model R²: {model_full.score(X_full, y_full):.4f}")
print(f"Gender coefficient ({sex_col_full}): ${full_gender_coef:,.2f}")
print(f"Interpretation: after controlling for occupation and industry, men earn ${full_gender_coef:,.2f} more than women.")

## Step 3: Wage Gap Decomposition Summary

Compares the two gender coefficients to calculate what % of the gap is explained vs. unexplained.

In [ ]:
explained_amount = baseline_gender_coef - full_gender_coef
explained_pct = (explained_amount / baseline_gender_coef) * 100
unexplained_pct = 100 - explained_pct

print("=" * 60)
print("GENDER WAGE GAP DECOMPOSITION")
print("=" * 60)
print(f"Raw gender gap (baseline):              ${baseline_gender_coef:,.2f}")
print(f"Adjusted gender gap (with occ/ind):     ${full_gender_coef:,.2f}")
print(f"Explained by occupation/industry:       ${explained_amount:,.2f} ({explained_pct:.1f}%)")
print(f"Unexplained gap:                        {unexplained_pct:.1f}%")
print()
print("Interpretation: Only 9.4% of the gender wage gap is explained by")
print("occupational/industry segregation. 90.6% persists within the same")
print("broad job type — suggesting the gap is not simply about job choice.")

## Step 4: Random Forest Regressor

Uses a 200,000-row sample (from 1.49M) for computational efficiency on a 2-core EC2 instance.  
100 trees, max depth 10. Trains a non-linear model to confirm gender's importance  
as a wage predictor without any assumption about a fixed linear effect.

In [ ]:
# ── Random Forest results computed by PySpark MLlib ──────────────
# Full pipeline in analysis/ml_rf_antara.py
# Run: python3 analysis/ml_rf_antara.py
# Results saved to data/processed/ and loaded below

import pandas as pd
from pathlib import Path

PROCESSED = Path("../data/processed")

rf_summary = pd.read_csv(PROCESSED / "rf_model_summary_antara.csv")
rf_imp = pd.read_csv(PROCESSED / "rf_feature_importance_antara.csv")
pdp = pd.read_csv(PROCESSED / "rf_partial_dependence_age_gender_antara.csv")

print("PySpark Random Forest Results:")
print(rf_summary.to_string(index=False))
print(f"\nTop 5 features:")
print(rf_imp.head(5).to_string(index=False))

## Step 5: Feature Importance and Partial Dependence

Feature importance shows which features were most useful for predicting wage across all 100 trees.  
Partial dependence shows predicted wage by age, split by gender, holding all other features at baseline.

## Step 6: Save Precomputed Results

Saves all results as small CSVs to `data/processed/` so `pages/ml_methods.qmd`  
can load them without retraining models on every site render.